# 02 — Maintenance-strategy model

Build, train and inspect the historical SAP strategy classifier.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)

In [ ]:
from ml_wartungsplan.data.build_strategy_dataset import build_strategy_dataset
from ml_wartungsplan.models.strategy import train_strategy_model
from ml_wartungsplan.settings import load_settings, resolve_project_path

settings = load_settings()
paths = settings["paths"]

In [ ]:
dataset = build_strategy_dataset(
    resolve_project_path(paths["raw_excel"]),
    resolve_project_path(paths["strategy_dataset"]),
)

print("Rows:", len(dataset))
display(dataset.head())
display(dataset["target_strategy"].value_counts().to_frame("rows"))

In [ ]:
metrics = train_strategy_model(
    dataset_path=resolve_project_path(paths["strategy_dataset"]),
    model_path=resolve_project_path(paths["strategy_model"]),
    report_dir=resolve_project_path(paths["reports_dir"]) / "strategy",
    random_state=settings["project"]["random_state"],
    test_folds=settings["strategy_model"]["test_folds"],
    tfidf_max_features=settings["strategy_model"]["tfidf_max_features"],
    min_document_frequency=settings["strategy_model"]["min_document_frequency"],
)

{k: v for k, v in metrics.items() if k != "classification_report"}

In [ ]:
predictions = pd.read_csv(
    PROJECT_ROOT / "reports/strategy/test_predictions.csv"
)
mistakes = predictions[
    predictions["actual_strategy"] != predictions["predicted_strategy"]
].sort_values("prediction_confidence", ascending=False)

print("Mistakes:", len(mistakes))
display(mistakes.head(40))